# RTX Spark local LLM setup

This notebook walks you through running a model on your laptop using
[NVIDIA's Local AI Inference Recipes](https://nvidia.github.io/local-ai-inference-recipes/hardware/rtx-spark/index.html)
([catalog source](https://github.com/NVIDIA/local-ai-inference-recipes)).

You tell it **your laptop's specs**, **which model** you want and **which engine** to use. It then runs each
setup step in order. Every step **checks whether it's already done first** and skips it if so, so you can
re-run the whole notebook at any time.

| Step | What it does | Already-done check |
|---|---|---|
| 0 | Your inputs | – |
| 1 | Detect laptop specs | – |
| 2 | Load the recipe catalog | Cached copy is less than 24 h old |
| 3 | Pick the recipe that fits your memory | – |
| 4 | Install llama.cpp (default `~/llama-cuda/<build>`, or a custom location) | `llama-server` exists and its build is new enough |
| 5 | Download model files into your models directory | Files exist with the expected size |
| 6 | Build the launch command and save it as a `.ps1` script | Script exists with the same content |
| 7 | Start `llama-server` | `/health` already returns `ok` |
| 8 | Send a test prompt | – |
| 9 | Stop the server | – |

> ⚠️ Recipes are provided by NVIDIA "for informational purposes only". Review each recipe's prerequisites,
> limitations and security notes (printed in step 3) before you run it.

## Step 0 – Your inputs
Edit these values, then run the notebook from top to bottom.

In [ ]:
import os
from pathlib import Path

# --- Laptop specs -----------------------------------------------------------
HARDWARE = "rtx-spark"        # hardware id from the catalog (e.g. "rtx-spark", "geforce-rtx", "rtx-pro")
OS_NAME = "windows"           # "windows" or "linux"
SYSTEM_MEMORY_GB = None       # total RAM / unified memory, e.g. 128 or 64. None = detect automatically
OS_RESERVE_GB = 16            # memory left for Windows + apps when choosing a recipe
MEMORY_BUDGET_GB = None       # memory the model may use. None = SYSTEM_MEMORY_GB - OS_RESERVE_GB

# --- Model + engine ---------------------------------------------------------
MODEL = "qwen/qwen3.8-flash-next"  # model id or display name (step 3 lists everything available)
BACKEND = "llama-cpp"              # inference engine: "llama-cpp" (or "vllm" on hardware that has vLLM recipes)
PRECISION = None                   # optional, e.g. "iq4-xs" or "q4-k-m"
VARIANT = None                     # optional, e.g. "vision-mtp" or "vision-dflash"
RECIPE_ID = None                   # optional: pin an exact recipe id from the table in step 3

# --- Locations --------------------------------------------------------------
LLAMA_CPP_DIR = Path.home() / "llama-cuda"   # llama.cpp builds go to LLAMA_CPP_DIR/<build>, e.g. ~/llama-cuda/b11617
LLAMA_SERVER_PATH = None                     # use an existing llama-server(.exe) instead of installing one
MODELS_DIR = Path.home() / "llm-models"      # model files go to MODELS_DIR/<hf-org>/<hf-repo>/<file>

# --- Server -----------------------------------------------------------------
HOST = None   # None = use the recipe's host (127.0.0.1)
PORT = None   # None = use the recipe's port

# --- Optional tokens (read from environment variables, never hard-code them) ---
HF_TOKEN = os.environ.get("HF_TOKEN")          # needed only for gated Hugging Face repos
GITHUB_TOKEN = os.environ.get("GITHUB_TOKEN")  # raises the GitHub API rate limit for the llama.cpp download

## Step 1 – Detect laptop specs

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))
import rtx_spark_setup as rss

system = rss.detect_system()
print(f"OS / arch : {system['os']} / {system['arch']}")
print(f"Python    : {system['python']}")
print(f"Memory    : {system['total_memory_gb']} GB")
for gpu in system["gpus"] or [{"name": "none detected (nvidia-smi not found)", "driver": "-", "memory": "-"}]:
    print(f"GPU       : {gpu['name']} | driver {gpu['driver']} | memory {gpu['memory']}")

if SYSTEM_MEMORY_GB is None:
    SYSTEM_MEMORY_GB = system["total_memory_gb"]
if MEMORY_BUDGET_GB is None and SYSTEM_MEMORY_GB:
    MEMORY_BUDGET_GB = max(SYSTEM_MEMORY_GB - OS_RESERVE_GB, 0)
print(f"\nUsing SYSTEM_MEMORY_GB={SYSTEM_MEMORY_GB}, MEMORY_BUDGET_GB={MEMORY_BUDGET_GB}")

if system["os"] != OS_NAME:
    print(f"⚠️  This machine runs {system['os']}, but OS_NAME is {OS_NAME!r}.")
if not system["gpus"]:
    print("⚠️  No NVIDIA GPU detected. Install the latest NVIDIA driver before continuing.")

## Step 2 – Load the recipe catalog

In [ ]:
catalog = rss.load_catalog()  # pass refresh=True to force a fresh download
print(f"{sum(1 for _ in rss.iter_recipes(catalog))} recipes for {len(catalog['models'])} models")

## Step 3 – Pick the recipe
Lists every recipe for your hardware, OS and engine that fits your memory budget, then picks the best match for `MODEL`.

In [ ]:
fits = rss.find_recipes(catalog, HARDWARE, OS_NAME, BACKEND, memory_budget_gb=MEMORY_BUDGET_GB)
backends = sorted({r["backend"] for _, r in rss.find_recipes(catalog, HARDWARE, OS_NAME)})
print(f"Engines available for {HARDWARE}/{OS_NAME}: {', '.join(backends) or 'none'}")
print(f"Recipes that fit in {MEMORY_BUDGET_GB} GB:\n")
print(rss.recipe_table(fits) if fits else "  (none)")

candidates = rss.find_recipes(catalog, HARDWARE, OS_NAME, BACKEND, MODEL, PRECISION, VARIANT, MEMORY_BUDGET_GB)
if not candidates:
    too_big = rss.find_recipes(catalog, HARDWARE, OS_NAME, BACKEND, MODEL, PRECISION, VARIANT)
    hint = f" Smallest recipe for this model needs {min(r['min_vidmem_gb'] for _, r in too_big)} GB." if too_big else ""
    raise SystemExit(f"No {BACKEND} recipe for {MODEL!r} on {HARDWARE}/{OS_NAME} fits {MEMORY_BUDGET_GB} GB.{hint}")

model_info, recipe = rss.choose_recipe(candidates, RECIPE_ID)
print(f"\n➡️  Selected: {recipe['id']}")
print(f"   {model_info['display_name']} – {model_info.get('summary', '')}")
print(f"   status={recipe['status']}  min memory={recipe['min_vidmem_gb']} GB  engine={recipe['backend']} {recipe.get('backend_version')}")
for title, key in [("Prerequisites", "prerequisites"), ("Limitations", "limitations")]:
    for item in recipe.get(key) or []:
        print(f"   {title}: {item}")
for key in ("security", "notes"):
    if recipe.get(key):
        print(f"   {key.title()}: {recipe[key]}")

## Step 4 – Install the engine (llama.cpp)
Installs the exact build the recipe was validated with into `LLAMA_CPP_DIR/<build>`. Skipped if that build (or a newer one meeting the recipe's minimum) is already there, or if you set `LLAMA_SERVER_PATH`.

In [ ]:
if recipe["backend"] != "llama-cpp":
    runtime = recipe.get("runtime") or {}
    print("Automated setup currently supports llama.cpp only. Run these recipe commands yourself:\n")
    print("# Setup\n" + (runtime.get("setup_command") or "(none)"))
    print("\n# Launch\n" + (recipe.get("serve") or {}).get("command", ""))
    raise SystemExit("Stopping: non-llama.cpp engine selected.")

if LLAMA_SERVER_PATH:
    llama_server = Path(LLAMA_SERVER_PATH).expanduser()
    if not llama_server.exists():
        raise FileNotFoundError(llama_server)
    print(f"✅ already done  using {llama_server} (build {rss.llama_server_version(llama_server)})")
else:
    llama_server = rss.install_llama_cpp(recipe, LLAMA_CPP_DIR, github_token=GITHUB_TOKEN)

## Step 5 – Download the model files
Downloads the target model (all shards), the speculative-decoding draft/MTP head and the vision projector the recipe uses into `MODELS_DIR`. Files already present with the right size are skipped; partial downloads resume.

In [ ]:
model_files = rss.model_files_for_recipe(recipe)
for f in model_files:
    print(f"  {f.role:<6} {f.repo_id}/{f.filename} @ {f.revision}")
print()
local_files = rss.download_model_files(model_files, MODELS_DIR, token=HF_TOKEN)

## Step 6 – Build the launch command
The recipe's command is kept as-is, except the Hugging Face download flags are swapped for the local files from step 5. It's also saved as a PowerShell script so you can start the server later without this notebook.

In [ ]:
plan = rss.build_launch_command(recipe, llama_server, local_files, host=HOST, port=PORT)
script = plan.powershell()
print(script)

script_path = Path(MODELS_DIR).expanduser() / "launch" / (recipe["id"].replace("/", "_") + ".ps1")
if script_path.exists() and script_path.read_text(encoding="utf-8") == script + "\n":
    print(f"\n✅ already done  {script_path}")
else:
    script_path.parent.mkdir(parents=True, exist_ok=True)
    script_path.write_text(script + "\n", encoding="utf-8")
    print(f"\n✅ done  saved {script_path}")

## Step 7 – Start the server
Starts `llama-server` in the background and waits until it has loaded the model. Large models can take several minutes. Skipped if a server is already healthy on the same port.

In [ ]:
log_file = Path(MODELS_DIR).expanduser() / "logs" / "llama-server.log"
server_proc = rss.start_server(plan, log_file)

## Step 8 – Test the endpoint
The server exposes an OpenAI-compatible API at the URL below.

In [ ]:
print(f"Endpoint: {plan.base_url}/v1")
print(rss.chat(plan.base_url, "Hello! In one sentence, who are you?"))

In [ ]:
# Use from any OpenAI client (pip install openai):
print(f"""from openai import OpenAI
client = OpenAI(base_url="{plan.base_url}/v1", api_key="EMPTY")
resp = client.chat.completions.create(model="local", messages=[{{"role": "user", "content": "Hello"}}])
print(resp.choices[0].message.content)""")

## Step 9 – Stop the server
The server keeps running so you can use it from other apps. Set `STOP_SERVER = True` and run this cell when you're finished. To start it again later without this notebook, run the script saved in step 6 from PowerShell.

In [ ]:
STOP_SERVER = False

if STOP_SERVER:
    rss.stop_server(server_proc)
else:
    print(f"Server left running at {plan.base_url} (set STOP_SERVER = True to stop it).")